# Four-mode GKP Hamiltonian: flux spectra and KITE asymmetry

calculates the ground-state transition frequencies

$$
f_{0k}=\frac{E_k-E_0}{h}
$$

of the four-mode Gridium Hamiltonian.

For each parameter regime, it reproduces the five sweep types:

1. Main-loop flux $\varphi_{\mathrm{ext}}$ with $\vartheta_{\mathrm{ext}}=0$.
2. Main-loop flux $\varphi_{\mathrm{ext}}$ with $\vartheta_{\mathrm{ext}}=\pi$.
3. KITE flux $\vartheta_{\mathrm{ext}}$ with $\varphi_{\mathrm{ext}}=0$.
4. KITE flux $\vartheta_{\mathrm{ext}}$ with $\varphi_{\mathrm{ext}}=\pi$.
5. Offset charge $n_g$ at the protected point
   $(\varphi_{\mathrm{ext}},\vartheta_{\mathrm{ext}})=(0,\pi)$.

- Solid lines: symmetric four-mode circuit. 
- Dashed lines:  same hamiltonian with Josephson and inductive KITE asymmetry.

## load and prepare the environment ##

In [ ]:
import os
import sys
from pathlib import Path

#each eigensolve should use one numerical-library thread.
#parallelism will be applied across independent sweep points.
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/gridium-mpl")

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

#find the repository root whether Jupyter starts in the repository root or inside the Notebooks directory.
ROOT = Path.cwd().resolve()

if not (ROOT / "Circuit_Objs").is_dir():
    for candidate in ROOT.parents:
        if (candidate / "Circuit_Objs").is_dir():
            ROOT = candidate
            break
    else:
        raise RuntimeError(
            "The notebook must be run from inside the 2q_gridium repository."
        )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from Circuit_Objs.qchard_gridium_netlist import Gridium4Mode

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 220,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})

print("Repository root:", ROOT)

## physical parameters and numerical cutoff ##

In [ ]:
#from Simulations.four_mode_gkp.four_mode_s5_asymmetry_spectra import N_TRANSITIONS
from Circuit_Objs.qchard_gridium_netlist import Gridium4Mode

PRESET  = "preview"
WORKERS = 2
SAVE_FIGURE = True 

PRESETS = {

    "preview": {
        "cutoffs": {
            "n1max": 3,
            "N2": 31,
            "L2": 11.0,
            "N3": 31,
            "L3": 14.0,
            "N4": 6,
            "nkeep": 40,
        },
        "flux_points": 9,
        "charge_points": 7,
    },

    "high_cutoff": {
        "cutoffs": {
            "n1max": 5,
            "N2": 131,
            "L2": 11.0,
            "N3": 101,
            "L3": 14.0,
            "N4": 8,
            "nkeep": 360,
        },
        "flux_points": 25,
        "charge_points": 21,
    },

}

CONFIG = PRESETS[PRESET]

#get the energies to get frequencies
N_TRANSITIONS = 5
N_LEVELS = N_TRANSITIONS + 1

#additional charging energies in four-mode hamiltonian 
FOUR_MODE_CAPS = {
    "eC": 5.5, 
    "eP": 10.0, 
}

#KITE asymmetry
ASYMMETRY = {
    "eps_J": 0.10, 
    "eps_LK": 0.05, 
}

#regimes a-d
REGIMES = {
    "a": {
        "EJ": 5.0, "EC": 0.5,
        "EL": 1.0, "ELK": 1.0,
        "EJS": 4.0, "ECS": 8.0,
    },
    "b": {
        "EJ": 10.0, "EC": 0.5,
        "EL": 1.0, "ELK": 1.0,
        "EJS": 4.0, "ECS": 8.0,
    },
    "c": {
        "EJ": 10.0, "EC": 0.5,
        "EL": 0.5, "ELK": 0.5,
        "EJS": 4.0, "ECS": 8.0,
    },
    "d": {
        "EJ": 10.0, "EC": 0.5,
        "EL": 0.2, "ELK": 0.2,
        "EJS": 4.0, "ECS": 8.0,
    },
}

RESULTS_DIR = ROOT / "Simulations" / "four_mode_gkp" / "results"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Preset:", PRESET)
print("Cutoffs:", CONFIG["cutoffs"])
print("Workers:", WORKERS)
print("Asymmetry:", ASYMMETRY)

## build the cached parallel evaluator ##

In [ ]:
import json
import hashlib
import time

from pathos.pools import ProcessPool

FORCE_RECOMPUTE = False

CACHE_PATH = (
    RESULTS_DIR
    / f"four_mode_s5_cache_{PRESET}.json"
)

print("Cache file:", CACHE_PATH)

## computes the simulation and saves to JSON cache ##

In [ ]:
#helper function converting numpy objects into ordinary python values 
def _plain(value):
    if isinstance(value, dict):
        return {
            str(key): _plain(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [_plain(item) for item in value]

    if isinstance(value, np.ndarray):
        return value.tolist()

    if isinstance(value, (np.floating, np.integer)):
        return value.item()

    return value

#creates a unique key for one hamiltonian parameter point
def _case_key(case):
    payload = json.dumps(
        _plain(case),
        sort_keys=True,
        separators=(",", ":"),
    )

    return hashlib.sha256(
        payload.encode("utf-8")
    ).hexdigest()[:20]

#calculates the transition frequency for one parameter point
def _solve_case(case):

    model = Gridium4Mode(**case)

    energies = np.asarray(
        model.levels(nlev=case["nlev"]),
        dtype=float,
    )

    transitions = energies - energies[0]

    return transitions.tolist()


#writes completed simulation results into a JSON cache 
def _write_cache(cache):
    temporary_path = CACHE_PATH.with_suffix(".tmp")

    temporary_path.write_text(
        json.dumps(cache, sort_keys=True)
    )

    temporary_path.replace(CACHE_PATH)

def evaluate_cases (cases, workers = 1, batch_size = 12): 
    cases = [_plain(case) for case in cases] #normalization key 
    keys = [_case_key(case) for case in cases] #converts every case into JSON values with unique keys 

    if CACHE_PATH.exists() and not FORCE_RECOMPUTE: 
        cache = json.loads(CACHE_PATH.read_text())
    else: 
        cache = {}
    
    unique_missing = {}

    for key, case in zip(keys, cases):
        if key not in cache: 
            unique_missing[key] = case 

    missing_items = list(unique_missing.items())

    print(
        f"Requested {len(cases)} points; "
        f"{len(missing_items)} unique points are missing."
    )

    #if at least one requested parameters point is absent from the cache 
    if missing_items: 
        started = time.time()

        if workers > 1: 
            pool = ProcessPool(
                nodes = min(workers, len(missing_items))
            )
        else: 
            pool = None

        try: #worker processes can be cleaned up if a calculation failed
            for start in range (
                0, 
                len(missing_items), 
                batch_size, 
            ): 
                batch = missing_items[
                    start: start + batch_size
                ]

                batch_cases = [
                    case
                    for _, case in batch 
                ]

                if pool is not None: 
                    values = list (
                        pool.map(_solve_case, batch_cases) #distribute the hamiltonian cases across worker process
                    )
                else: 
                    values = [
                        _solve_case(case)
                        for case in batch_cases
                    ]

                for (key, _), value in zip(batch, values): #updating cache
                    cache[key]  = value 

                _write_cache(cache)

                #reports progress after every completed and cached batch 
                completed = min(
                    start + len(batch), 
                    len(missing_items), 
                )

                elapsed = time.time() - started 

                print(
                    f"Solved {completed}/{len(missing_items)} "
                    f"missing points in {elapsed / 60:.1f} minutes."
                )

        #cleanup  
        except BaseException: 
            if pool is not None: 
                pool.terminate()
            raise 

        else: 
            if pool is not None: 
                pool.close()
                pool.join()
        
        finally: 
            if pool is not None: 
                pool.clear()
        
    return np.asarray(
        [cache[key] for key in keys],  #final shape: (N_cases, N_levels)
        dtype = float, 
    )


## sweep setup and definitions ##

In [ ]:
phi_values = np.linspace (
    0.0, 
    2.0 * np.pi, 
    CONFIG["flux_points"]
)

theta_values = np.linspace ( # KITE-loop flux 
    0.0, 
    2.0 * np.pi, 
    CONFIG["flux_points"]
)

ng_values = np.linspace(
    0.0, 
    1.0, 
    CONFIG["charge_points"]
)

cases = [] # hold every complete hamiltonian parameter dictionary 
registry = {} #record which rows of cases belong to each plotted curve 

#creates on complete spectrum series 
def register_series (
    label, 
    base_parameters, 
    parameter, 
    values, 
    fixed_parameters, 
    asymmetry,
): 
    indices = []

    for value in values: 
        case = dict(base_parameters)

        case.update(FOUR_MODE_CAPS)
        case.update(CONFIG["cutoffs"])
        case.update(fixed_parameters)
        case.update(asymmetry)

        case[parameter] = float(value)
        case['nlev'] = N_LEVELS

        indices.append(len(cases))
        cases.append(case)

    registry[label] = np.asarray(
        indices, 
        dtype = int, 
    )

#panel definitions 
PANEL_SPECS = {

    "phi_theta0": (
        "phi_ext", 
        phi_values, 
        {
            "theta_ext": 0.0, 
            "phi_ext": 0.0, 
            "ng": 0.0, 
        }, 
    ), 

    "phi_thetapi": ( #sweep phi while keep theta as pi 
        "phi_ext", 
        phi_values, 
        {
            "theta_ext": np.pi, 
            "phi_ext": 0.0, 
            "ng": 0.0, 
        }, 
    ),

    "theta_phi0": (
        "theta_ext", 
        theta_values, 
        {
            "phi_ext": 0.0, 
            "theta_ext": 0.0, 
            "ng": 0.0, 
        }, 
    ), 

    "theta_phipi": (
        "theta_ext", 
        theta_values, 
        {
            "phi_ext": np.pi, 
            "theta_ext": 0.0, 
            "ng": 0.0, 
        }, 
    ), 

    "charge": (
        "ng", 
        ng_values, 
        {
            "phi_ext": 0.0, 
            "theta_ext": np.pi, 
            "ng": 0.0, 
        }, 
    ), 
}

In [ ]:
for regime_name, regime_parameters in REGIMES.items(): #iterate through a-d
    for panel_name, panel_spec in PANEL_SPECS.items(): #iterate through five panels 
        parameter, values, fixed_parameters = panel_spec

        #create one graph for symmetric, one graph for asymmetric 
        register_series (
            label=f"{regime_name}:{panel_name}:symmetric", 
            base_parameters= regime_parameters, 
            parameter = parameter, 
            values = values, 
            fixed_parameters=fixed_parameters, 
            asymmetry = {
                "eps_J": 0.0, 
                "eps_LK": 0.0, 
            }, 
        )

        register_series (
            label=f"{regime_name}:{panel_name}:asymmetric", 
            base_parameters=regime_parameters, 
            parameter = parameter, 
            values = values, 
            fixed_parameters=fixed_parameters, 
            asymmetry = ASYMMETRY, 
        )

print("Registered parameter points:", len(cases))
print("Registered spectrum series:", len(registry))

In [ ]:
#diagonalization
energy_table = evaluate_cases (
    cases, 
    workers=WORKERS, 
)

In [ ]:
#reorganize the rows into named spectrum curves
spectra = {
    label: energy_table[indices]
    for label, indices in registry.items()
}

#consistency check 
assert energy_table.shape == (len(cases), N_LEVELS)
assert np.all(np.isfinite(energy_table))

assert np.allclose( #should return energies relative to the ground state 
    energy_table[:, 0],
    0.0,
    atol=1e-10,
)

print("Energy table shape:", energy_table.shape)
print("Number of spectrum series:", len(spectra))

## plotting ##

In [ ]:
transition_colors = plt.get_cmap("tab10").colors[:N_TRANSITIONS]

row_names = list(REGIMES)
spectrum_panels = ["phi_theta0", "phi_thetapi", "theta_phi0", "theta_phipi"]

panel_titles = [
    r"$\varphi_{\mathrm{ext}}$ sweep, $\vartheta_{\mathrm{ext}}=0$",
    r"$\varphi_{\mathrm{ext}}$ sweep, $\vartheta_{\mathrm{ext}}=\pi$",
    r"$\vartheta_{\mathrm{ext}}$ sweep, $\varphi_{\mathrm{ext}}=0$",
    r"$\vartheta_{\mathrm{ext}}$ sweep, $\varphi_{\mathrm{ext}}=\pi$",
]


def format_flux_axis(ax, symbol):
    ax.set_xlim(0.0, 2.0 * np.pi)
    ax.set_xticks([0.0, np.pi, 2.0 * np.pi], ["0", r"$\pi$", r"$2\pi$"])
    ax.set_xlabel(symbol)
    ax.grid(alpha=0.22)


fig = plt.figure(figsize=(20, 15))
outer = fig.add_gridspec(
    4, 5, width_ratios=[1.0, 1.0, 1.0, 1.0, 0.82],
    left=0.055, right=0.985, bottom=0.055, top=0.90,
    wspace=0.34, hspace=0.48,
)

for row_index, regime_name in enumerate(row_names):
    row_max = 0.0
    for panel_name in spectrum_panels:
        for variant in ("symmetric", "asymmetric"):
            spectrum = spectra[f"{regime_name}:{panel_name}:{variant}"]
            row_max = max(row_max, float(np.max(spectrum[:, 1:])))

    for column_index, panel_name in enumerate(spectrum_panels):
        ax = fig.add_subplot(outer[row_index, column_index])
        symmetric = spectra[f"{regime_name}:{panel_name}:symmetric"]
        asymmetric = spectra[f"{regime_name}:{panel_name}:asymmetric"]
        x = phi_values if column_index < 2 else theta_values

        for level in range(1, N_LEVELS):
            color = transition_colors[level - 1]
            ax.plot(x, symmetric[:, level], color=color, linewidth=1.45)
            ax.plot(
                x, asymmetric[:, level], color=color,
                linewidth=1.25, linestyle="--",
            )

        ax.set_ylim(0.0, 1.04 * row_max)
        flux_label = (
            r"External flux, $\varphi_{\mathrm{ext}}$"
            if column_index < 2
            else r"KITE flux, $\vartheta_{\mathrm{ext}}$"
        )
        format_flux_axis(ax, flux_label)

        if column_index == 0:
            ax.set_ylabel("Transition frequency (GHz)")
            ax.text(
                -0.25, 1.08, regime_name, transform=ax.transAxes,
                fontsize=16, fontweight="bold",
            )
        if row_index == 0:
            ax.set_title(panel_titles[column_index], fontsize=11)

    charge_grid = outer[row_index, 4].subgridspec(N_TRANSITIONS, 1, hspace=0.10)
    charge_symmetric = spectra[f"{regime_name}:charge:symmetric"]
    charge_asymmetric = spectra[f"{regime_name}:charge:asymmetric"]
    center = int(np.argmin(np.abs(ng_values - 0.5)))

    for level in range(1, N_LEVELS):
        ax = fig.add_subplot(charge_grid[level - 1, 0])
        symmetric_dispersion = 1e3 * np.abs(
            charge_symmetric[:, level] - charge_symmetric[center, level]
        )
        asymmetric_dispersion = 1e3 * np.abs(
            charge_asymmetric[:, level] - charge_asymmetric[center, level]
        )
        color = transition_colors[level - 1]
        ax.plot(ng_values, symmetric_dispersion, color=color, linewidth=1.35)
        ax.plot(
            ng_values, asymmetric_dispersion, color=color,
            linewidth=1.15, linestyle="--",
        )
        ymax = max(
            float(np.max(symmetric_dispersion)),
            float(np.max(asymmetric_dispersion)),
            0.01,
        )
        ax.set_xlim(0.0, 1.0)
        ax.set_ylim(0.0, 1.10 * ymax)
        ax.set_yticks([0.0, ymax])
        ax.tick_params(axis="both", labelsize=7, pad=1)
        ax.text(
            0.02, 0.82, rf"$f_{{0{level}}}$",
            transform=ax.transAxes, color=color, fontsize=7, va="top",
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.65, pad=0.4),
        )
        ax.grid(alpha=0.18)
        if level < N_TRANSITIONS:
            ax.set_xticklabels([])
        else:
            ax.set_xticks([0.0, 0.5, 1.0])
            ax.set_xlabel(r"Offset charge, $n_g$", fontsize=9)
        if level == 3:
            ax.set_ylabel("Charge dispersion (MHz)", fontsize=9)
        if row_index == 0 and level == 1:
            ax.set_title("Charge dispersion", fontsize=11)

legend_handles = [
    Line2D(
        [0], [0], color=transition_colors[level - 1], linewidth=2,
        label=rf"$f_{{0{level}}}$",
    )
    for level in range(1, N_LEVELS)
]
legend_handles += [
    Line2D([0], [0], color="black", linewidth=1.8, label="symmetric"),
    Line2D(
        [0], [0], color="black", linewidth=1.8, linestyle="--",
        label=(
            rf"asymmetric: $\epsilon_J={100 * ASYMMETRY['eps_J']:.0f}\%$, "
            rf"$\epsilon_{{LK}}={100 * ASYMMETRY['eps_LK']:.0f}\%$"
        ),
    ),
]
fig.legend(
    handles=legend_handles, loc="upper center", ncol=len(legend_handles),
    bbox_to_anchor=(0.5, 0.945), frameon=False,
)
fig.suptitle(
    f"Fig. S5-style spectra from the handwritten four-mode Hamiltonian ({PRESET})",
    fontsize=17, y=0.982,
)

s5_path = RESULTS_DIR / f"four_mode_s5_style_{PRESET}.png"
if SAVE_FIGURE:
    fig.savefig(s5_path, bbox_inches="tight", dpi=200)
    print("Saved:", s5_path)

plt.show()


## cusp slope of the 0-1 transition at $\vartheta_{\mathrm{ext}}=\pi$ ##

At the KITE sweet spot $\vartheta_{\mathrm{ext}}=\pi$ (with $\varphi_{\mathrm{ext}}=0$, $n_g=0$)
the symmetric circuit has a level crossing between the two lowest states, so
$f_{01}$ has a **cusp**: $f_{01}\propto |\vartheta_{\mathrm{ext}}-\pi|$ close to
the crossing.  The one-sided slope

$$
s \;=\; \left.\frac{\partial f_{01}}{\partial\vartheta_{\mathrm{ext}}}\right|_{\vartheta_{\mathrm{ext}}\to\pi^{+}}
\;=\;
\langle 1|\,\partial_{\vartheta_{\mathrm{ext}}} H\,|1\rangle
-\langle 0|\,\partial_{\vartheta_{\mathrm{ext}}} H\,|0\rangle
$$

is extracted with the Hellmann–Feynman operator
[`Gridium4Mode.d_theta()`](../Circuit_Objs/qchard_gridium_netlist.py) evaluated at
$\vartheta_{\mathrm{ext}}=\pi+\delta$ with a small $\delta$ (the slope is
converged to $<0.1\%$ for $\delta\le 0.05$, see printout below).

Physically $s/2\pi$ is the difference in **persistence current** circulating in
the KITE loop between the two crossing states, so we expect it to be set by the
inductive energy scale of the KITE, $E_{LK}$, rather than by the ratio
$E_J/E_{LK}$ alone.

In [ ]:
#cusp-slope helpers: cached evaluation of f01 and its theta-derivative
CUSP_CACHE_PATH = RESULTS_DIR / "four_mode_cusp_slope_cache.json"

#same "preview" cutoffs as above, but only two levels are needed
CUSP_BASE = dict(
    EJ=10.0, EC=0.5, EL=1.0, ELK=1.0, EJS=4.0, ECS=8.0,
    eC=5.5, eP=10.0, eps_J=0.0, eps_LK=0.0,
    ng=0.0, phi_ext=0.0, theta_ext=float(np.pi), nlev=2,
    n1max=3, N2=31, L2=11.0, N3=31, L3=14.0, N4=6, nkeep=40,
)

CUSP_DELTA = 0.03  #one-sided offset: theta_ext = pi + CUSP_DELTA

def _cusp_solve(case):
    """Return [f01, d f01/d theta_ext] via the Hellmann-Feynman operator."""
    model = Gridium4Mode(**case)
    energies = np.asarray(model.levels(nlev=2), dtype=float)
    dth = np.real(np.diag(model.d_theta(nlev=2).full()))
    return [float(energies[1] - energies[0]), float(dth[1] - dth[0])]

if CUSP_CACHE_PATH.exists():
    _cusp_cache = json.loads(CUSP_CACHE_PATH.read_text())
else:
    _cusp_cache = {}
    print("Cusp cache not found: points will be solved on demand (~3 s each).")

def cusp_point(**overrides):
    """(f01, d f01/d theta_ext) for CUSP_BASE parameters + overrides."""
    case = _plain({**CUSP_BASE, **overrides})
    key = _case_key(case)
    if key not in _cusp_cache:
        _cusp_cache[key] = _cusp_solve(case)
        CUSP_CACHE_PATH.write_text(json.dumps(_cusp_cache, sort_keys=True))
    f01, slope = _cusp_cache[key]
    return float(f01), float(slope)

def cusp_slope(**overrides):
    overrides.setdefault("theta_ext", float(np.pi + CUSP_DELTA))
    return cusp_point(**overrides)[1]

print(f"Cusp cache: {len(_cusp_cache)} cached points")
print("delta convergence of the one-sided slope (EJ=10, ELK=1):")
for _d in [0.01, 0.02, 0.03, 0.05, 0.08]:
    print(f"  delta = {_d:5.2f}:  d f01/d theta = "
          f"{cusp_point(theta_ext=float(np.pi + _d))[1]:.4f} GHz/rad")

In [ ]:
#visualize the cusp and the extracted one-sided slope
theta_scan = np.linspace(np.pi - 0.6, np.pi + 0.6, 25)
scan_combos = [(10.0, 1.0), (5.0, 1.0), (10.0, 0.5)]

fig, axes = plt.subplots(1, len(scan_combos), figsize=(13, 4), sharex=True)
for ax, (EJ, ELK) in zip(axes, scan_combos):
    f01 = np.array([cusp_point(EJ=EJ, ELK=ELK, theta_ext=float(t))[0]
                    for t in theta_scan])
    ax.plot(theta_scan, f01, "o-", ms=3.5, lw=1.2, color="tab:blue")

    #one-sided Hellmann-Feynman slope evaluated just above theta = pi
    f0, s0 = cusp_point(EJ=EJ, ELK=ELK, theta_ext=float(np.pi + CUSP_DELTA))
    tt = np.linspace(np.pi, np.pi + 0.45, 20)
    ax.plot(tt, f0 + s0 * (tt - (np.pi + CUSP_DELTA)), "--", color="tab:red",
            lw=1.6, label=rf"HF slope = {s0:.3f} GHz/rad")

    ax.axvline(np.pi, color="gray", lw=0.8, alpha=0.6)
    ax.set_xticks([np.pi - 0.5, np.pi, np.pi + 0.5],
                  [r"$\pi-0.5$", r"$\pi$", r"$\pi+0.5$"])
    ax.set_xlabel(r"KITE flux $\vartheta_{\mathrm{ext}}$")
    ax.set_title(rf"$E_J={EJ:g}$, $E_{{LK}}={ELK:g}$ GHz")
    ax.legend(fontsize=8, loc="lower left")
axes[0].set_ylabel(r"$f_{01}$ (GHz)")
fig.suptitle(r"Cusp of $f_{01}$ at $\vartheta_{\mathrm{ext}}=\pi$ "
             r"($\varphi_{\mathrm{ext}}=0$, $n_g=0$)")
fig.tight_layout()
if SAVE_FIGURE:
    fig.savefig(RESULTS_DIR / "cusp_f01_vs_theta.png", dpi=200,
                bbox_inches="tight")
plt.show()

### slope versus circuit-energy ratios ###

The panels below sweep $E_J$ and $E_{LK}$ (and, as controls, $E_L$ and $E_C$):

- **(A)** The raw slope is **not** a single-valued function of $E_J/E_{LK}$:
  curves at different $E_{LK}$ do not collapse, so the naive guess
  $s = s(E_J/E_{LK})$ fails.
- **(B)** At fixed (large) $E_J$ the slope is almost exactly **linear in
  $E_{LK}$** (fitted power $\approx 0.9$), approaching the analytic
  fluxonium-like limit $s \to \pi E_{LK}$: when $E_J \gg E_{LK}$ the two
  crossing states sit in adjacent wells of the KITE potential separated by
  $\Delta\theta \simeq 2\pi$, and the inductive energy
  $\tfrac{1}{2}E_{LK}(\theta - \vartheta_{\mathrm{ext}})^2$ then gives
  $s = E_{LK}\,\Delta\theta/2 \simeq \pi E_{LK}$.
- **(C)** The dimensionless combination $s/E_{LK}$ *does* collapse reasonably
  onto a function of $E_J/E_{LK}$: it rises from $\lesssim 1$ for
  $E_J/E_{LK}\lesssim 3$ (well phase-slips soften the crossing and shrink the
  persistence-current difference) and saturates near $\pi$ (with a slow
  logarithmic-like drift from well-shape corrections) for
  $E_J/E_{LK}\gtrsim 6$.
- **(D)** Controls: varying the main-loop $E_L$ or the junction $E_C$ changes
  $s/E_{LK}$ only weakly in the transmon-like regime $E_J/E_C\gg 1$ ($E_C$
  matters only once $E_J/E_C$ becomes small, where phase slips proliferate).

**Summary:** the cusp slope is governed by
$s \approx \pi E_{LK}\, g(E_J/E_{LK})$ with $g \to 1$ for
$E_J/E_{LK} \gtrsim 6$ — i.e. the meaningful analytic relationship is a linear
dependence on $E_{LK}$ with a saturation function of the ratio $E_J/E_{LK}$,
not a dependence on $E_J/E_{LK}$ alone.

In [ ]:
#slope as a function of circuit-energy ratios
EJ_vals = [3.0, 4.0, 6.0, 8.0, 10.0, 13.0, 16.0, 20.0]
ELK_vals = [0.35, 0.5, 0.7, 1.0, 1.4, 2.0]

grid = np.array([
    (EJ, ELK, cusp_slope(EJ=EJ, ELK=ELK))
    for EJ in EJ_vals for ELK in ELK_vals
])

#power-law fit of slope vs ELK in the deep-EJ regime (EJ = 20)
deep = grid[grid[:, 0] == 20.0]
p_elk = np.polyfit(np.log(deep[:, 1]), np.log(deep[:, 2]), 1)
print(f"deep-EJ fit: slope ~ ELK^{p_elk[0]:.2f} "
      f"(prefactor {np.exp(p_elk[1]):.2f}, cf. pi = {np.pi:.2f})")

fig, axes = plt.subplots(2, 2, figsize=(11, 8.5))

#(A) raw slope vs EJ/ELK: curves at different ELK do not collapse
ax = axes[0, 0]
for ELK in ELK_vals:
    sub = grid[grid[:, 1] == ELK]
    ax.plot(sub[:, 0] / ELK, sub[:, 2], "o-", ms=4,
            label=rf"$E_{{LK}}={ELK:g}$")
ax.set_xlabel(r"$E_J/E_{LK}$")
ax.set_ylabel(r"$|df_{01}/d\vartheta_{\mathrm{ext}}|_{\pi^+}$ (GHz/rad)")
ax.set_title("(A) raw slope vs $E_J/E_{LK}$: no collapse")
ax.legend(fontsize=8)

#(B) slope vs ELK at fixed EJ: linear in ELK, approaching pi*ELK
ax = axes[0, 1]
for EJ in [6.0, 10.0, 20.0]:
    sub = grid[grid[:, 0] == EJ]
    ax.loglog(sub[:, 1], sub[:, 2], "o-", ms=4, label=rf"$E_J={EJ:g}$")
xx = np.array([0.3, 2.3])
ax.loglog(xx, np.exp(p_elk[1]) * xx ** p_elk[0], "k--", lw=1,
          label=rf"fit: $\propto E_{{LK}}^{{{p_elk[0]:.2f}}}$")
ax.loglog(xx, np.pi * xx, ":", color="gray", lw=1.2,
          label=r"$\pi E_{LK}$ (fluxonium limit)")
ax.set_xlabel(r"$E_{LK}$ (GHz)")
ax.set_ylabel("slope (GHz/rad)")
ax.set_title("(B) slope vs $E_{LK}$ at fixed $E_J$")
ax.legend(fontsize=8)

#(C) dimensionless collapse: slope/ELK vs EJ/ELK
ax = axes[1, 0]
for ELK in ELK_vals:
    sub = grid[grid[:, 1] == ELK]
    ax.semilogx(sub[:, 0] / ELK, sub[:, 2] / ELK, "o", ms=4,
                label=rf"$E_{{LK}}={ELK:g}$")
ax.axhline(np.pi, color="gray", ls=":", lw=1.2)
ax.text(0.98, np.pi + 0.06, r"$\pi$", color="gray", fontsize=10,
        transform=ax.get_yaxis_transform(), ha="right")
ax.set_xlabel(r"$E_J/E_{LK}$")
ax.set_ylabel(r"slope$/E_{LK}$ (rad$^{-1}$)")
ax.set_title("(C) collapse: slope$/E_{LK}$ vs $E_J/E_{LK}$")
ax.legend(fontsize=8)

#(D) weak dependence on the other circuit energies
ax = axes[1, 1]
EL_vals = [0.2, 0.5, 1.0, 2.0]
for ELK in [0.5, 1.0, 2.0]:
    s_el = [cusp_slope(EL=EL, ELK=ELK) for EL in EL_vals]
    ax.plot(np.array(EL_vals) / ELK, np.array(s_el) / ELK, "s-", ms=4,
            label=rf"$E_L$ sweep, $E_{{LK}}={ELK:g}$")
EC_vals = [0.25, 0.5, 1.0]
for EJ, ELK in [(10.0, 1.0), (4.0, 1.0)]:
    s_ec = [cusp_slope(EJ=EJ, EC=EC, ELK=ELK) for EC in EC_vals]
    ax.plot(np.array(EC_vals), np.array(s_ec) / ELK, "^--", ms=4,
            label=rf"$E_C$ sweep, $E_J={EJ:g}$")
ax.set_xlabel(r"$E_L/E_{LK}$  (squares)   or   $E_C$ in GHz  (triangles)")
ax.set_ylabel(r"slope$/E_{LK}$ (rad$^{-1}$)")
ax.set_title("(D) sensitivity to $E_L$ and $E_C$")
ax.legend(fontsize=7)

fig.suptitle(r"One-sided slope of $f_{01}$ at "
             r"$\vartheta_{\mathrm{ext}}\to\pi^+$", fontsize=13)
fig.tight_layout()
if SAVE_FIGURE:
    fig.savefig(RESULTS_DIR / "cusp_slope_vs_ratios.png", dpi=200,
                bbox_inches="tight")
plt.show()